In [1]:
print('Loading Packages...\n')
%matplotlib osx
import sys, os          
import uproot
import matplotlib.pyplot as plt
import matplotlib as mpl
from tqdm import trange
import numpy as np
from scipy.optimize import curve_fit
import pandas
from collections import defaultdict
import scipy.stats
import matplotlib.gridspec as gridspec
from scipy.stats import chi2_contingency
import random
from matplotlib.ticker import (MultipleLocator, AutoMinorLocator)
font = {'family' : 'serif', 'size' : 14 }
mpl.rc('font', **font)
mpl.rcParams['mathtext.fontset'] = 'cm' # Set the math font to Computer Modern
mpl.rcParams['legend.fontsize'] = 16
# Set default figure and axes face colors to white
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

# make sure this file is present in your pwd
df = pandas.read_csv('FullTankPMTGeometry.csv')
channel_number = []; PMT_type = []
for i in range(len(df['channel_num'])):
    channel_number.append(df['channel_num'][i])
    PMT_type.append(df['PMT_type'][i])
    
dead_mask = {416, 343, 358, 359, 408, 366}  # updated PMT dead mask to reflect data
                                            # in case you haven't filtered out PMTs in ToolAnalysis

print('done')

Loading Packages...

done


### AmBe, data

In [10]:
# adjust if needed --> output from extract_neutron_candidates.py:
# https://github.com/S81D/AmBe_neutron_candidates/blob/main/extract_neutron_candidates.py
data_file = '../AmBe/AmBe_neutron_candidates/neutron_candidates_v5.root'

# This is based on AmBe v2.0 (summer - fall 2023)
# see Steven Doran dissertation for more details

# ----------------------------------------------------- #

cluster_time = []; cluster_charge = []; cluster_QB = []; cluster_hits = []; 
hit_times = []; hit_charges = []; hit_ids = []
source_position = [[], [], []]      # x, y, z

# ----------------------------------------------------- #

# last index in each list are for the counts
poss = [[0,100,0,0],[0,50,0,0],[0,0,0,0],[0,-50,0,0],[0,-100,0,0],            # Port 5
       [0,100,-75,0],[0,50,-75,0],[0,0,-75,0],[0,-50,-75,0],[0,-100,-75,0],   # Port 1
       [75,100,0,0],[75,50,0,0],[75,0,0,0],[75,-50,0,0],[75,-100,0,0],        # Port 4
       [0,100,102,0],[0,50,102,0],[0,0,102,0],[0,-50,102,0],[0,-100,102,0]]   # Port 3


print('\nLoading AmBe event data...')

neg_hits = []; neg_charge = []
with uproot.open(data_file) as file:

    Event = file["Neutrons"]
    CT = Event["cluster_time"].array()
    CPE = Event["cluster_charge"].array()
    CCB = Event["cluster_Qb"].array()
    CH = Event["cluster_hits"].array()
    hT = Event["hitT"].array()
    hPE = Event["hitPE"].array()
    hID = Event["hitID"].array()
    xp = Event["X_pos"].array()
    yp = Event["Y_pos"].array()
    zp = Event["Z_pos"].array()

    for i in trange(len(CT)):

        x1 = round(xp[i]); y1 = round(yp[i]); z1 = round(zp[i])
        
        # [0,100,102,0] (remove Port 3 +100cm)
        # AmBe v2.0 did not take proper data at this location --> omit it
        if y1 == 100 and x1 == 0 and z1 == 102:
            continue
        
        # ENABLE for Type-only study
        '''
        cc = 0; ch = 0
        for j in range(len(hPE[i])):
            indy = hID[i][j] - 332
            if PMT_type[indy] == 'Watchboy':
                cc += hPE[i][j]
                ch += 1
        if ch > 0:  # enable to remove 0 bin
            cluster_charge.append(cc)   
            cluster_hits.append(ch)
        #'''
        
        # if you want to apply the dead mask
        cpe_sum = sum(hPE[i][j] for j in range(len(hID[i])) if hID[i][j] not in dead_mask)
        ch_sum = sum(1 for j in range(len(hID[i])) if hID[i][j] not in dead_mask)
        cluster_charge.append(cpe_sum)
        cluster_hits.append(ch_sum)
        
        # indent if above is uncommented ^ 
        cluster_time.append(CT[i])   
        #cluster_charge.append(CPE[i])   # comment if using Type-only
        cluster_QB.append(CCB[i])
        #cluster_hits.append(CH[i])      # comment if using Type-only
        hit_times.append(hT[i])
        hit_charges.append(hPE[i])
        hit_ids.append(hID[i])

        # In AmBe v2.0 (2023), Port 4 is backwards in X
        if x1 == -75:
            source_position[0].append(75)
        else:
            source_position[0].append(x1)
        source_position[1].append(y1)
        source_position[2].append(z1)

    
print('----------------------------------------------------------------\n')
print('We have: ', len(cluster_time), ' total AmBe neutron candidates\n')

print('\ndone')


Loading AmBe event data...


100%|████████████████████████████████████| 67940/67940 [02:39<00:00, 426.20it/s]

----------------------------------------------------------------

We have:  65992  total AmBe neutron candidates


done


### AmBe, MC

In [11]:
# adjust if needed
MC_directory_approx_tilt = '../AmBe/AmBe_MC/pmt_tilting_v1/corrected_waveforms/QE_1.50_WB_ETEL_LUX_1.0_HM_WM_1.25_corrected_waveforms/'
file_names_approx_tilt = [file for file in os.listdir(MC_directory_approx_tilt) if os.path.isfile(os.path.join(MC_directory_approx_tilt, file))]
QE_tag_approx_tilt = '_QE_1.50_WB_ETEL_LUX_1.0_HM_WM_1.25'
    
    
cluster_charge_MC = []; source_position_MC = [[], [], []]; cluster_hits_MC = []
    
for file in file_names_approx_tilt:

    root = uproot.open(MC_directory_approx_tilt + file)
    T = root['phaseIITankClusterTree']
    CEN = T['eventNumber'].array()
    CPE = T['clusterPE'].array()
    CCB = T['clusterChargeBalance'].array()
    CH = T['clusterHits'].array()
    CT = T['clusterTime'].array()
    PE1 = T['hitPE'].array()
    ID1 = T['hitDetID'].array()
    HT1 = T['hitT'].array()
    CN = T['clusterNumber'].array()

    # Port 5
    if file == 'MC_AmBe_port5_z100' + QE_tag_approx_tilt + '.root':
        xp = 0; yp = 100; zp = 0;
    elif file == 'MC_AmBe_port5_z50' + QE_tag_approx_tilt + '.root':
        xp = 0; yp = 50; zp = 0;
    elif file == 'MC_AmBe_port5_z0' + QE_tag_approx_tilt + '.root':
        xp = 0; yp = 0; zp = 0; 
    elif file == 'MC_AmBe_port5_zminus50' + QE_tag_approx_tilt + '.root':
        xp = 0; yp = -50; zp = 0; 
    elif file == 'MC_AmBe_port5_zminus100' + QE_tag_approx_tilt + '.root':
        xp = 0; yp = -100; zp = 0; 

    # Port 1
    elif file == 'MC_AmBe_port1_z100' + QE_tag_approx_tilt + '.root':
        xp = 0; yp = 100; zp = -75; 
    elif file == 'MC_AmBe_port1_z50' + QE_tag_approx_tilt + '.root':
        xp = 0; yp = 50; zp = -75; 
    elif file == 'MC_AmBe_port1_z0' + QE_tag_approx_tilt + '.root':
        xp = 0; yp = 0; zp = -75; 
    elif file == 'MC_AmBe_port1_zminus50' + QE_tag_approx_tilt + '.root':
        xp = 0; yp = -50; zp = -75; 
    elif file == 'MC_AmBe_port1_zminus100' + QE_tag_approx_tilt + '.root':
        xp = 0; yp = -100; zp = -75; 

    # Port 4
    elif file == 'MC_AmBe_port4_z100' + QE_tag_approx_tilt + '.root':
        xp = 75; yp = 100; zp = 0; 
    elif file == 'MC_AmBe_port4_z50' + QE_tag_approx_tilt + '.root':
        xp = 75; yp = 50; zp = 0; 
    elif file == 'MC_AmBe_port4_z0' + QE_tag_approx_tilt + '.root':
        xp = 75; yp = 0; zp = 0; 
    elif file == 'MC_AmBe_port4_zminus50' + QE_tag_approx_tilt + '.root':
        xp = 75; yp = -50; zp = 0; 
    elif file == 'MC_AmBe_port4_zminus100' + QE_tag_approx_tilt + '.root':
        xp = 75; yp = -100; zp = 0; 

    # Port 3
    elif file == 'MC_AmBe_port3_z100' + QE_tag_approx_tilt + '.root':
        #xp = 0; yp = 100; zp = 102; 
        continue    # skip to reflect data
    elif file == 'MC_AmBe_port3_z50' + QE_tag_approx_tilt + '.root':
        xp = 0; yp = 50; zp = 102; 
    elif file == 'MC_AmBe_port3_z0' + QE_tag_approx_tilt + '.root':
        xp = 0; yp = 0; zp = 102;
    elif file == 'MC_AmBe_port3_zminus50' + QE_tag_approx_tilt + '.root':
        xp = 0; yp = -50; zp = 102; 
    elif file == 'MC_AmBe_port3_zminus100' + QE_tag_approx_tilt + '.root':
        xp = 0; yp = -100; zp = 102; 


    # in case you mess up the file name, we don't want to populate xp, yp, zp with the arrays from the AmBe data cell
    else:
        print('INCORRECT FILE NAME!!!')


    # APPLY SELECTION CUTS (same as data)
    for i in trange(len(CPE), desc = file):

        # selection cuts (timing includes latency)
        if CPE[i] < 70 and (67000-1101) > CT[i] > (2000-1101) and CCB[i] < 0.4:

            # only cluster in the event (can adjust this for multiplicity / background study)
            if CN[i] != 0:
                continue
            else:    # zeroth cluster, make sure its the only cluster
                if i + 1 < len(CN):
                    if CN[i + 1] != 0:
                        continue     

            ''' comment for PMT-type analysis
            cc = 0
            for j in range(len(PE1[i])):
                indy = ID1[i][j] - 332
                if PMT_type[indy] == 'Watchboy':
                    cc += PE1[i][j]
            if cc > 0:       # for zero bin
            #'''
            
            # if you want to apply the dead mask
            cpe_sum = sum(PE1[i][j] for j in range(len(ID1[i])) if ID1[i][j] not in dead_mask)
            ch_sum = sum(1 for j in range(len(ID1[i])) if ID1[i][j] not in dead_mask)
            cluster_charge_MC.append(cpe_sum)
            cluster_hits_MC.append(ch_sum)

                # requires indentation if doing PMT-type analysis
            #cluster_charge_MC.append(CPE[i])  # comment for type-only
            #cluster_charge_MC[geo].append(cc)     # uncomment for type-only
            #cluster_hits_MC.append(CH[i])
            source_position_MC[0].append(xp)
            source_position_MC[1].append(yp)
            source_position_MC[2].append(zp)


print('\ndone')

MC_AmBe_port3_zminus50_QE_1.50_WB_ETEL_LUX_1.0_HM_WM_1.25.root: 100%|█| 15365/15
MC_AmBe_port1_zminus50_QE_1.50_WB_ETEL_LUX_1.0_HM_WM_1.25.root: 100%|█| 17164/17
MC_AmBe_port1_z0_QE_1.50_WB_ETEL_LUX_1.0_HM_WM_1.25.root: 100%|█| 17280/17280 [0
MC_AmBe_port3_z0_QE_1.50_WB_ETEL_LUX_1.0_HM_WM_1.25.root: 100%|█| 15515/15515 [0
MC_AmBe_port5_zminus50_QE_1.50_WB_ETEL_LUX_1.0_HM_WM_1.25.root: 100%|█| 17465/17
MC_AmBe_port3_zminus100_QE_1.50_WB_ETEL_LUX_1.0_HM_WM_1.25.root: 100%|█| 13539/1
MC_AmBe_port4_z50_QE_1.50_WB_ETEL_LUX_1.0_HM_WM_1.25.root: 100%|█| 17422/17422 [
MC_AmBe_port1_zminus100_QE_1.50_WB_ETEL_LUX_1.0_HM_WM_1.25.root: 100%|█| 15916/1
MC_AmBe_port1_z50_QE_1.50_WB_ETEL_LUX_1.0_HM_WM_1.25.root: 100%|█| 17458/17458 [
MC_AmBe_port5_z100_QE_1.50_WB_ETEL_LUX_1.0_HM_WM_1.25.root: 100%|█| 17670/17670 
MC_AmBe_port4_z0_QE_1.50_WB_ETEL_LUX_1.0_HM_WM_1.25.root: 100%|█| 17326/17326 [0
MC_AmBe_port4_z100_QE_1.50_WB_ETEL_LUX_1.0_HM_WM_1.25.root: 100%|█| 17147/17147 
MC_AmBe_port3_z50_QE_1.50_WB


done


In [17]:
# this snippet will produce a cumulative PMT charge plot and provide chi-sq agreements

plot_save_dir = 'plots/geometry_comparison/AmBe/approx_geo_agreement/'
plot_save_name_base = 'QE_1.50_HM_1.25_Dead_PMTs_implemented'

verbose = True    # enable for verbosity

# --- AmBe plot details ---- #
geo_colors = 'dodgerblue'
bin_size_AmBe = 1.0                             # default: 1.0 (0.5 for Watchboys / HM)
binning_AmBe = np.arange(5, 70, bin_size_AmBe)  # default: 5, 70 (0:30 for WB / HM)
xlim_low_AmBe = 0; xlim_high_AmBe = 75          # plot xlim window: default for all: [0,75] (0:35 for WB / HM)

# # # # # # # # # # # # # # # # # # # # # # # # # #

# useful functions

# see Steven Doran dissertation for more details (in particular the chi2 appendix):
# - for the AmBe neutrons, we randomly sample each port position, depth
#     to build a randomized MC / Data sample to compare, rather than scaling the total entries

def get_common_positions(d1, d2):
    common_keys = set(d1.keys()) & set(d2.keys())
    filtered = []
    for k in common_keys:
        if len(d1[k]) > 0 and len(d2[k]) > 0:
            filtered.append(k)
    return filtered

def balance_dataset_with_keys(pos_dict, keys, n_per_pos):
    all_entries = []
    for k in sorted(keys):
        if len(pos_dict[k]) >= n_per_pos:
            all_entries.extend(random.sample(pos_dict[k], n_per_pos))
    return all_entries

def generate_merged_bins(bins, counts_data, counts_mc_1, min_counts=10):
    """
    Given initial bin edges and counts for data and MC, merge adjacent bins
    until each has at least `min_counts` in both histograms.

    Returns a new list of bin edges (fill_bins).
    """
    assert len(bins) == len(counts_data) + 1, "Bin edge mismatch"

    new_bins = [bins[0]]  # always start with the first edge
    i = 0
    while i < len(counts_data):
        # Start with current bin
        sum_d = counts_data[i]
        sum_mc_1 = counts_mc_1[i]
        j = i + 1

        # Merge forward until we meet minimum count requirement
        while (sum_d < min_counts or sum_mc_1 < min_counts and j < len(counts_data)):
            sum_d += counts_data[j]
            sum_mc_1 += counts_mc_1[j]
            j += 1

        # Append right edge of merged bin
        new_bins.append(bins[j])
        i = j  # Move to next unprocessed bin

    return new_bins


def balance_samples(list1, list2):
    """
    Randomly samples both lists to ensure the same number of entries.
    Returns the balanced versions.
    """
    list1 = list(list1)
    list2 = list(list2)
    n = min(len(list1), len(list2))
    return random.sample(list1, n), random.sample(list2, n)
    
    
# no Port 3 +100cm data was loaded in, so it should not be included. 
# "mc_keys_approx_tilt" has only 19 entries, so it looks good.
    
# # # # # # # # # # # # # # # # # # # # # # # # # #

data_dict = defaultdict(list)
mc_dict_approx_tilt = defaultdict(list)

# Prepare full sample
data_all = []; mc_all_approx_tilt = []

# we must ensure equal counts at each position source for all MC and data

# Key: (x, y, z)
for i in range(len(cluster_charge)):
    key = (source_position[0][i], source_position[1][i], source_position[2][i])
    data_dict[key].append(cluster_charge[i])

for i in range(len(cluster_charge_MC)):
    key = (source_position_MC[0][i], source_position_MC[1][i], source_position_MC[2][i])
    mc_dict_approx_tilt[key].append(cluster_charge_MC[i])

mc_keys_approx_tilt = list(mc_dict_approx_tilt.keys())
data_keys = list(data_dict.keys())

# Find shared, valid keys
valid_keys = get_common_positions(data_dict, mc_dict_approx_tilt)

# Get minimum across all datasets for just those positions
min_data = min(len(data_dict[k]) for k in valid_keys)
min_mc_approx_tilt = min(len(mc_dict_approx_tilt[k]) for k in valid_keys)
min_common = min(min_data, min_mc_approx_tilt)

# Now sample equally per position from these valid keys
data_all = balance_dataset_with_keys(data_dict, valid_keys, min_common)
mc_all_approx_tilt = balance_dataset_with_keys(mc_dict_approx_tilt, valid_keys, min_common)

# Sanity check
assert len(data_all) == len(mc_all_approx_tilt), "Data and MC not balanced after enforced key matching!"

if verbose:
    print('\nCounts after sampling:', len(data_all), len(mc_all_approx_tilt), '\n')


# MC counts and bins
counts_data, bins_data = np.histogram(data_all, bins=binning_AmBe)
raw_counts_mc_approx_tilt, bins_mc_approx_tilt = np.histogram(mc_all_approx_tilt, bins=binning_AmBe)

counts_mc_approx_tilt = raw_counts_mc_approx_tilt

fill_bins = generate_merged_bins(binning_AmBe, counts_data,counts_mc_approx_tilt, min_counts=10)

# re-histogram with updated binning
counts_data, bins_data = np.histogram(data_all, bins=fill_bins)
raw_counts_mc_approx_tilt, bins_mc_approx_tilt = np.histogram(mc_all_approx_tilt, bins=fill_bins)

counts_mc_approx_tilt = raw_counts_mc_approx_tilt

if verbose:
    print("\nMerged bin edges:")
    for i in range(len(fill_bins) - 1):
        print(f"Bin {i}: [{fill_bins[i]}, {fill_bins[i+1]})")


# Errors
xwidth = [(fill_bins[i+1]-fill_bins[i])/2 for i in range(len(fill_bins)-1)]
data_errors = np.sqrt(counts_data)
mc_errors_approx_tilt = np.sqrt(raw_counts_mc_approx_tilt)
binscenters = np.array([0.5 * (fill_bins[i] + fill_bins[i+1]) for i in range(len(fill_bins)-1)])


# Compute chi-squared (combined MC + Data errors)
chi2_approx_tilt = np.sum(((np.array(counts_data) - np.array(counts_mc_approx_tilt)) ** 2) / 
          (np.array(data_errors) ** 2 + np.array(mc_errors_approx_tilt) ** 2))
ndf = len(counts_data) - 1  # degrees of freedom (can be adjusted)
if verbose:
    print(f"\nApprox tilt Chi2/NDF: {chi2_approx_tilt / ndf:.2f}")
    

# calculate residuals and errors
mc_approx_tilt_data = []; mc_errors_approx_tilt_plot = []; 
x_info = []
        
for i in range(len(counts_data)):
    if counts_data[i] == 0:
        continue  # skip bin with no data
    x_info.append(binscenters[i])
        
    # MC (approx-tilted)
    if counts_mc_approx_tilt[i] == 0:
        mc_approx_tilt_data.append(0)
        mc_errors_approx_tilt_plot.append(0)
    else:
        ratio_mc_approx_tilt = counts_mc_approx_tilt[i] / counts_data[i]
        error_mc_approx_tilt = ratio_mc_approx_tilt * np.sqrt(
            (data_errors[i] / counts_data[i])**2 +
            (mc_errors_approx_tilt[i] / counts_mc_approx_tilt[i])**2
        )
        mc_approx_tilt_data.append(ratio_mc_approx_tilt)
        mc_errors_approx_tilt_plot.append(error_mc_approx_tilt)
        

# # # # # # # # # # # # # # # # # # # # # # # # # #
# plotting
# # # # # # # # # # # # # # # # # # # # # # # # # #

fig = plt.figure(figsize=(6, 5))
gs = fig.add_gridspec(2, 1, height_ratios=[3, 1])  # 2 rows, 1 column
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])

ax1.set_title('MC / Data AmBe neutrons', fontsize = 16)
ax2.set_xlabel('Cluster charge [p.e.]', fontsize = 14)    # FIX
ax1.set_ylabel(f'clusters / {bin_size_AmBe} p.e. bin', fontsize = 14)
ax2.set_ylabel('MC / Data', fontsize = 14)

ax1.set_xlim([xlim_low_AmBe, xlim_high_AmBe]); ax2.set_xlim([xlim_low_AmBe, xlim_high_AmBe]); ax2.set_ylim([0, 2]); 

# MC histogram (approx-tilted)
ax1.hist(binscenters, bins=fill_bins, weights=counts_mc_approx_tilt, histtype='step', color=geo_colors, label='MC', linewidth=2)
ax1.fill_between(      # shaded MC error band (turn off if needed)
    binscenters,
    counts_mc_approx_tilt - mc_errors_approx_tilt,
    counts_mc_approx_tilt + mc_errors_approx_tilt,
    step='mid',
    color=geo_colors,
    alpha=0.3
)

# data histogram
ax1.errorbar(binscenters, counts_data, yerr=data_errors, fmt='o', color='k', label='Data')

# subplot below for residuals
ax2.errorbar(x_info, mc_approx_tilt_data, xerr=xwidth, yerr=mc_errors_approx_tilt_plot, markersize = 1,
             fmt='s', color=geo_colors, label = 'MC')
ax2.axhline(1, linestyle = 'dashed', color = 'k')

ax1.text(0.67, 0.52, f"events = {sum(counts_data)}", transform = ax1.transAxes, fontsize=10)
ax1.text(0.67, 0.42, f"$\chi^2$/ndf = {round(chi2_approx_tilt)} / {int(ndf)}", transform = ax1.transAxes, fontsize=10, color='k')
ax1.legend(fontsize = 12, frameon = False, loc = 'upper right')
#ax2.legend(fontsize = 10, frameon = False, loc = 'upper right')

plt.tight_layout()
path_str = plot_save_dir + 'MC_Data_AmBe_neutrons_ApproxTilt_cluster_pe_' + plot_save_name_base + '.png'
#plt.savefig(path_str,dpi=300,bbox_inches='tight',pad_inches=.3,facecolor = 'w')
plt.show()

print('\ndone')


Counts after sampling: 28785 28785 


Merged bin edges:
Bin 0: [5.0, 6.0)
Bin 1: [6.0, 7.0)
Bin 2: [7.0, 8.0)
Bin 3: [8.0, 9.0)
Bin 4: [9.0, 10.0)
Bin 5: [10.0, 11.0)
Bin 6: [11.0, 12.0)
Bin 7: [12.0, 13.0)
Bin 8: [13.0, 14.0)
Bin 9: [14.0, 15.0)
Bin 10: [15.0, 16.0)
Bin 11: [16.0, 17.0)
Bin 12: [17.0, 18.0)
Bin 13: [18.0, 19.0)
Bin 14: [19.0, 20.0)
Bin 15: [20.0, 21.0)
Bin 16: [21.0, 22.0)
Bin 17: [22.0, 23.0)
Bin 18: [23.0, 24.0)
Bin 19: [24.0, 25.0)
Bin 20: [25.0, 26.0)
Bin 21: [26.0, 27.0)
Bin 22: [27.0, 28.0)
Bin 23: [28.0, 29.0)
Bin 24: [29.0, 30.0)
Bin 25: [30.0, 31.0)
Bin 26: [31.0, 32.0)
Bin 27: [32.0, 33.0)
Bin 28: [33.0, 34.0)
Bin 29: [34.0, 35.0)
Bin 30: [35.0, 36.0)
Bin 31: [36.0, 37.0)
Bin 32: [37.0, 38.0)
Bin 33: [38.0, 39.0)
Bin 34: [39.0, 40.0)
Bin 35: [40.0, 41.0)
Bin 36: [41.0, 42.0)
Bin 37: [42.0, 43.0)
Bin 38: [43.0, 44.0)
Bin 39: [44.0, 45.0)
Bin 40: [45.0, 46.0)
Bin 41: [46.0, 47.0)
Bin 42: [47.0, 48.0)
Bin 43: [48.0, 49.0)
Bin 44: [49.0, 50.0)
Bin 45: [50.0, 51

In [33]:
# this snippet will produce a cumulative PMT hits plot

plot_save_dir = 'plots/geometry_comparison/AmBe/approx_geo_agreement/'
plot_save_name_base = 'QE_1.50_HM_1.25_Dead_PMTs_implemented'

verbose = True    # enable for verbosity

# --- AmBe ---- #
geo_colors = 'dodgerblue'
bin_size_AmBe = 2.0
binning_AmBe = np.arange(6, 36, bin_size_AmBe)    # default: 5, 70 (0:30 for WB / HM)
xlim_low_AmBe = 6; xlim_high_AmBe = 40    # plot xlim window: default for all: [0,75] (0:35 for WB / HM)

# # # # # # # # # # # # # # # # # # # # # # # # # #

# useful functions

def get_common_positions(d1, d2):
    common_keys = set(d1.keys()) & set(d2.keys())
    filtered = []
    for k in common_keys:
        if len(d1[k]) > 0 and len(d2[k]) > 0:
            filtered.append(k)
    return filtered

def balance_dataset_with_keys(pos_dict, keys, n_per_pos):
    all_entries = []
    for k in sorted(keys):
        if len(pos_dict[k]) >= n_per_pos:
            all_entries.extend(random.sample(pos_dict[k], n_per_pos))
    return all_entries

def generate_merged_bins(bins, counts_data, counts_mc_1, min_counts=10):
    """
    Given initial bin edges and counts for data and MC, merge adjacent bins
    until each has at least `min_counts` in both histograms.

    Returns a new list of bin edges (fill_bins).
    """
    assert len(bins) == len(counts_data) + 1, "Bin edge mismatch"

    new_bins = [bins[0]]  # always start with the first edge
    i = 0
    while i < len(counts_data):
        # Start with current bin
        sum_d = counts_data[i]
        sum_mc_1 = counts_mc_1[i]
        j = i + 1

        # Merge forward until we meet minimum count requirement
        while (sum_d < min_counts or sum_mc_1 < min_counts and j < len(counts_data)):
            sum_d += counts_data[j]
            sum_mc_1 += counts_mc_1[j]
            j += 1

        # Append right edge of merged bin
        new_bins.append(bins[j])
        i = j  # Move to next unprocessed bin

    return new_bins


def balance_samples(list1, list2):
    """
    Randomly samples both lists to ensure the same number of entries.
    Returns the balanced versions.
    """
    list1 = list(list1)
    list2 = list(list2)
    n = min(len(list1), len(list2))
    return random.sample(list1, n), random.sample(list2, n)
    
    
# no Port 3 +100cm data was loaded in, so it should not be included. 
# "mc_keys_approx_tilt" has only 19 entries, so it looks good.
    
# # # # # # # # # # # # # # # # # # # # # # # # # #
# AmBe plot
# # # # # # # # # # # # # # # # # # # # # # # #
#print('\n------- AmBe plot -------\n')
data_dict = defaultdict(list)
mc_dict_approx_tilt = defaultdict(list)

# Prepare full sample
data_all = []; mc_all_approx_tilt = []

# we must ensure equal counts at each position source for all MC and data

# Key: (x, y, z)
for i in range(len(cluster_charge)):
    key = (source_position[0][i], source_position[1][i], source_position[2][i])
    data_dict[key].append(cluster_hits[i])

for i in range(len(cluster_charge_MC)):
    key = (source_position_MC[0][i], source_position_MC[1][i], source_position_MC[2][i])
    mc_dict_approx_tilt[key].append(cluster_hits_MC[i])

mc_keys_approx_tilt = list(mc_dict_approx_tilt.keys())
data_keys = list(data_dict.keys())

# Find shared, valid keys
valid_keys = get_common_positions(data_dict, mc_dict_approx_tilt)

# Get minimum across all datasets for just those positions
min_data = min(len(data_dict[k]) for k in valid_keys)
min_mc_approx_tilt = min(len(mc_dict_approx_tilt[k]) for k in valid_keys)
min_common = min(min_data, min_mc_approx_tilt)


# Now sample equally per position from these valid keys
data_all = balance_dataset_with_keys(data_dict, valid_keys, min_common)
mc_all_approx_tilt = balance_dataset_with_keys(mc_dict_approx_tilt, valid_keys, min_common)

# Sanity check
assert len(data_all) == len(mc_all_approx_tilt), "Data and MC not balanced after enforced key matching!"

if verbose:
    print('\nCounts after sampling:', len(data_all), len(mc_all_approx_tilt), '\n')


# MC counts and bins
counts_data, bins_data = np.histogram(data_all, bins=binning_AmBe)
raw_counts_mc_approx_tilt, bins_mc_approx_tilt = np.histogram(mc_all_approx_tilt, bins=binning_AmBe)

counts_mc_approx_tilt = raw_counts_mc_approx_tilt

fill_bins = generate_merged_bins(binning_AmBe, counts_data,counts_mc_approx_tilt, min_counts=10)

# re-histogram with updated binning
counts_data, bins_data = np.histogram(data_all, bins=fill_bins)
raw_counts_mc_approx_tilt, bins_mc_approx_tilt = np.histogram(mc_all_approx_tilt, bins=fill_bins)

counts_mc_approx_tilt = raw_counts_mc_approx_tilt

if verbose:
    print("\nMerged bin edges:")
    for i in range(len(fill_bins) - 1):
        print(f"Bin {i}: [{fill_bins[i]}, {fill_bins[i+1]})")


# Errors
xwidth = [(fill_bins[i+1]-fill_bins[i])/2 for i in range(len(fill_bins)-1)]
data_errors = np.sqrt(counts_data)
mc_errors_approx_tilt = np.sqrt(raw_counts_mc_approx_tilt)
binscenters = np.array([0.5 * (fill_bins[i] + fill_bins[i+1]) for i in range(len(fill_bins)-1)])


# Compute chi-squared
chi2_approx_tilt = np.sum(((np.array(counts_data) - np.array(counts_mc_approx_tilt)) ** 2) / 
          (np.array(data_errors) ** 2 + np.array(mc_errors_approx_tilt) ** 2))
ndf = len(counts_data) - 1  # degrees of freedom (can be adjusted)
if verbose:
    print(f"\nApprox tilt Chi2/NDF: {chi2_approx_tilt / ndf:.2f}")
    

# calculate residuals and errors
mc_approx_tilt_data = []; mc_errors_approx_tilt_plot = []; 
x_info = []
        
for i in range(len(counts_data)):
    if counts_data[i] == 0:
        continue  # skip bin with no data
    x_info.append(binscenters[i])
        
    # MC (approx-tilted)
    if counts_mc_approx_tilt[i] == 0:
        mc_approx_tilt_data.append(0)
        mc_errors_approx_tilt_plot.append(0)
    else:
        ratio_mc_approx_tilt = counts_mc_approx_tilt[i] / counts_data[i]
        error_mc_approx_tilt = ratio_mc_approx_tilt * np.sqrt(
            (data_errors[i] / counts_data[i])**2 +
            (mc_errors_approx_tilt[i] / counts_mc_approx_tilt[i])**2
        )
        mc_approx_tilt_data.append(ratio_mc_approx_tilt)
        mc_errors_approx_tilt_plot.append(error_mc_approx_tilt)
        

fig = plt.figure(figsize=(6, 5))
gs = fig.add_gridspec(2, 1, height_ratios=[3, 1])  # 2 rows, 1 column
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])

ax1.set_title('MC / Data AmBe neutrons', fontsize = 16)
ax2.set_xlabel('PMT Hits', fontsize = 14)    # FIX
ax1.set_ylabel(f'clusters / PMT hit bin', fontsize = 14)
ax2.set_ylabel('MC / Data', fontsize = 14)

ax1.set_xlim([xlim_low_AmBe, xlim_high_AmBe]); ax2.set_xlim([xlim_low_AmBe, xlim_high_AmBe]); ax2.set_ylim([0, 2]); 

# MC histogram (approx-tilted)
ax1.hist(binscenters, bins=fill_bins, weights=counts_mc_approx_tilt, histtype='step', color=geo_colors, label='MC', linewidth=2)
ax1.fill_between(      # shaded MC error band (turn off if needed)
    binscenters,
    counts_mc_approx_tilt - mc_errors_approx_tilt,
    counts_mc_approx_tilt + mc_errors_approx_tilt,
    step='mid',
    color=geo_colors,
    alpha=0.3
)

# data histogram
ax1.errorbar(binscenters, counts_data, yerr=data_errors, fmt='o', color='k', label='Data')

# subplot below for residuals
ax2.errorbar(x_info, mc_approx_tilt_data, xerr=xwidth, yerr=mc_errors_approx_tilt_plot, markersize = 1,
             fmt='s', color=geo_colors, label = 'MC')
ax2.axhline(1, linestyle = 'dashed', color = 'k')

ax1.text(0.67, 0.52, f"events = {sum(counts_data)}", transform = ax1.transAxes, fontsize=10)
ax1.text(0.67, 0.42, f"$\chi^2$/ndf = {round(chi2_approx_tilt)} / {int(ndf)}", transform = ax1.transAxes, fontsize=10, color='k')
ax1.legend(fontsize = 12, frameon = False, loc = 'upper right')
#ax2.legend(fontsize = 10, frameon = False, loc = 'upper right')

plt.tight_layout()
path_str = plot_save_dir + 'MC_Data_AmBe_neutrons_ApproxTilt_cluster_hits_' + plot_save_name_base + '.png'
plt.savefig(path_str,dpi=300,bbox_inches='tight',pad_inches=.3,facecolor = 'w')
plt.show()

print('\ndone')


Counts after sampling: 28785 28785 


Merged bin edges:
Bin 0: [6.0, 8.0)
Bin 1: [8.0, 10.0)
Bin 2: [10.0, 12.0)
Bin 3: [12.0, 14.0)
Bin 4: [14.0, 16.0)
Bin 5: [16.0, 18.0)
Bin 6: [18.0, 20.0)
Bin 7: [20.0, 22.0)
Bin 8: [22.0, 24.0)
Bin 9: [24.0, 26.0)
Bin 10: [26.0, 28.0)
Bin 11: [28.0, 30.0)
Bin 12: [30.0, 32.0)
Bin 13: [32.0, 34.0)

Approx tilt Chi2/NDF: 28.58

done


In [19]:
'''Same as above for charge but no plotting - just run the chi sq fitting N times for an average / median value'''

# rather than scaling the MC counts to data, we randomly sample. We should therefore perform an N
# trial chi-sq agreement to reduce shift / bias from MC stats

# place this snippet above the N hits to compare charge, leave it here and run N hits to compare N hits

total_x2_approx = []

verbose = False

# # # # # # # # # # # # # # # # # # # # # # # # # #

for jx in trange(1000):
    
    data_dict = defaultdict(list)
    mc_dict_approx_tilt = defaultdict(list)

    # Prepare full sample
    data_all = []; mc_all_approx_tilt = []

    # we must ensure equal counts at each position source for all MC and data

    # Key: (x, y, z)
    for i in range(len(cluster_charge)):
        key = (source_position[0][i], source_position[1][i], source_position[2][i])
        data_dict[key].append(cluster_charge[i])

    for i in range(len(cluster_charge_MC)):
        key = (source_position_MC[0][i], source_position_MC[1][i], source_position_MC[2][i])
        mc_dict_approx_tilt[key].append(cluster_charge_MC[i])

    mc_keys_approx_tilt = list(mc_dict_approx_tilt.keys())
    data_keys = list(data_dict.keys())

    # Find shared, valid keys
    valid_keys = get_common_positions(data_dict, mc_dict_approx_tilt)

    # Get minimum across all datasets for just those positions
    min_data = min(len(data_dict[k]) for k in valid_keys)
    min_mc_approx_tilt = min(len(mc_dict_approx_tilt[k]) for k in valid_keys)
    min_common = min(min_data, min_mc_approx_tilt)


    # Now sample equally per position from these valid keys
    data_all = balance_dataset_with_keys(data_dict, valid_keys, min_common)
    mc_all_approx_tilt = balance_dataset_with_keys(mc_dict_approx_tilt, valid_keys, min_common)

    # Sanity check
    assert len(data_all) == len(mc_all_approx_tilt), "Data and MC not balanced after enforced key matching!"

    if verbose:
        print('\nCounts after sampling:', len(data_all), len(mc_all_approx_tilt), '\n')


    # MC counts and bins
    counts_data, bins_data = np.histogram(data_all, bins=binning_AmBe)
    raw_counts_mc_approx_tilt, bins_mc_approx_tilt = np.histogram(mc_all_approx_tilt, bins=binning_AmBe)

    counts_mc_approx_tilt = raw_counts_mc_approx_tilt

    fill_bins = generate_merged_bins(binning_AmBe, counts_data, counts_mc_approx_tilt, min_counts=10)

    # re-histogram with updated binning
    counts_data, bins_data = np.histogram(data_all, bins=fill_bins)
    raw_counts_mc_approx_tilt, bins_mc_approx_tilt = np.histogram(mc_all_approx_tilt, bins=fill_bins)

    counts_mc_approx_tilt = raw_counts_mc_approx_tilt

    if verbose:
        print("\nMerged bin edges:")
        for i in range(len(fill_bins) - 1):
            print(f"Bin {i}: [{fill_bins[i]}, {fill_bins[i+1]})")


    # Errors
    xwidth = [(fill_bins[i+1]-fill_bins[i])/2 for i in range(len(fill_bins)-1)]
    data_errors = np.sqrt(counts_data)
    mc_errors_approx_tilt = np.sqrt(raw_counts_mc_approx_tilt)
    binscenters = np.array([0.5 * (fill_bins[i] + fill_bins[i+1]) for i in range(len(fill_bins)-1)])


    # Compute chi-squared
    chi2_approx_tilt = np.sum(((np.array(counts_data) - np.array(counts_mc_approx_tilt)) ** 2) / 
              (np.array(data_errors) ** 2 + np.array(mc_errors_approx_tilt) ** 2))
    ndf = len(counts_data) - 1  # degrees of freedom (can be adjusted)
    if verbose:
        print(f"\nApprox tilt Chi2/NDF: {chi2_approx_tilt / ndf:.2f}")

    total_x2_approx.append(chi2_approx_tilt / ndf)


print('\ndone')

100%|███████████████████████████████████████| 1000/1000 [02:13<00:00,  7.51it/s]


done


In [26]:
# Plot the chi2

print(np.median(total_x2_approx))

fig, ax1 = plt.subplots(figsize=(4, 3))
plt.hist(total_x2_approx, bins = 10, histtype = 'step', color = 'dodgerblue', label = 'Approx. geometry', linewidth = 2)
#plt.legend(fontsize = 12, frameon = False, loc = 'upper center')
plt.xlabel(r'$\chi^2$' + '/dof', loc = 'right')
plt.title('AmBe MC fit')
#plt.xlim([0,10.6])
#ax1.text(0.33, 0.64, f"$\chi^2$/ndf = {round(np.median(total_x2_default),2)}", transform = ax1.transAxes, fontsize=12, color='tab:orange')
ax1.text(0.58, 0.7, f"$\chi^2$/ndf = {round(np.median(total_x2_approx),2)}", transform = ax1.transAxes, fontsize=12, color='dodgerblue')
plt.tight_layout()
plt.savefig('plots/geometry_comparison/AmBe/approx_geo_agreement/chi2_fit_results_cumulative_charge_1000_trials_ApproxTilt.png',
            dpi=300,bbox_inches='tight',pad_inches=.3,facecolor = 'w')
plt.show()

1.7426694965370606


### compare by position

In [12]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.gridspec as gridspec
from scipy.stats import chi2_contingency
import random
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches

# [0,100,102,0] (remove Port 3 +100cm)

# labels to match source position (poss) list
labels = [
    "Port 5, Y = +100cm", "Port 5, Y = +50cm", "Port 5, Y = 0cm", "Port 5, Y = -50cm", "Port 5, Y = -100cm",
    "Port 1, Y = +100cm", "Port 1, Y = +50cm", "Port 1, Y = 0cm", "Port 1, Y = -50cm", "Port 1, Y = -100cm",
    "Port 4, Y = +100cm", "Port 4, Y = +50cm", "Port 4, Y = 0cm", "Port 4, Y = -50cm", "Port 4, Y = -100cm",
    "Port 3, Y = +100cm", "Port 3, Y = +50cm", "Port 3, Y = 0cm", "Port 3, Y = -50cm", "Port 3, Y = -100cm"
]

MC_poss = [[0,100,0,0],[0,50,0,0],[0,0,0,0],[0,-50,0,0],[0,-100,0,0],            # Port 5
          [0,100,-75,0],[0,50,-75,0],[0,0,-75,0],[0,-50,-75,0],[0,-100,-75,0],   # Port 1
          [75,100,0,0],[75,50,0,0],[75,0,0,0],[75,-50,0,0],[75,-100,0,0],        # Port 4
          [0,100,102,0], [0,50,102,0],[0,0,102,0],[0,-50,102,0],[0,-100,102,0]]  # Port 3


def generate_merged_bins(bins, counts_data, counts_mc_1, min_counts=10):
    """
    Given initial bin edges and counts for data and MC, merge adjacent bins
    until each has at least `min_counts` in both histograms.

    Returns a new list of bin edges (fill_bins).
    """
    assert len(bins) == len(counts_data) + 1, "Bin edge mismatch"

    new_bins = [bins[0]]  # always start with the first edge
    i = 0
    while i < len(counts_data):
        # Start with current bin
        sum_d = counts_data[i]
        sum_mc_1 = counts_mc_1[i]
        j = i + 1

        # Merge forward until we meet minimum count requirement
        while (sum_d < min_counts or sum_mc_1 < min_counts) and j < len(counts_data):
            sum_d += counts_data[j]
            sum_mc_1 += counts_mc_1[j]
            j += 1

        # Append right edge of merged bin
        new_bins.append(bins[j])
        i = j  # Move to next unprocessed bin

    return new_bins


geo_colors = 'dodgerblue'


# Define number of rows and columns for the grid layout
fig, axes = plt.subplots(4, 5, figsize=(20, 12))  # 3 rows, 5 columns

for sp in range(len(poss)):
    
    # default
    bin_size = 4
    binning = np.arange(5, 69, bin_size)
    
    # Watchboy/HM
    #bin_size = 2
    #binning = np.arange(0, 28, bin_size)

    print('\n\n############################')
    print(labels[sp])
    
    # Determine subplot location
    row = sp // 5
    col = sp % 5
    ax = axes[row, col]
    
    # we don't want a figure for Port 3 +100cm --> keep it blank
    if labels[sp] == "Port 3, Y = +100cm":
        ax.axis("off")  # keep blank
        continue
    
    # we can extract the full data and MC samples for this position
    data_ = [cluster_charge[i] for i in range(len(cluster_charge)) if 
             source_position[0][i] == poss[sp][0] and
             source_position[1][i] == poss[sp][1] and
             source_position[2][i] == poss[sp][2]]
    
    mc_approx_tilt_ = [cluster_charge_MC[i] for i in range(len(cluster_charge_MC)) if 
               source_position_MC[0][i] == MC_poss[sp][0] and 
               source_position_MC[1][i] == MC_poss[sp][1] and
               source_position_MC[2][i] == MC_poss[sp][2]]
    
    
    # then equalize entries per position by random sampling
    nmin = min(len(data_), len(mc_approx_tilt_))
    if nmin == 0:
        continue  # Skip if no valid entries

    # Randomly sample equal-sized subsets
    data_ = random.sample(data_, nmin)
    mc_approx_tilt_ = random.sample(mc_approx_tilt_, nmin)
    
    #print(len(data_), len(mc_no_tilt_), len(mc_indy_tilt_), len(mc_approx_tilt_))
    if len(data_) == 0 or len(mc_approx_tilt_) == 0:
        continue  # Skip if no data
    
    
    # MC counts and bins
    counts_data, bins_data = np.histogram(data_, bins=binning)
    raw_counts_mc_approx_tilt, bins_mc_approx_tilt = np.histogram(mc_approx_tilt_, bins=binning)

    counts_mc_approx_tilt = raw_counts_mc_approx_tilt

    fill_bins = generate_merged_bins(binning, counts_data, counts_mc_approx_tilt, min_counts=10)

    # re-histogram with updated binning
    counts_data, bins_data = np.histogram(data_, bins=fill_bins)
    raw_counts_mc_approx_tilt, bins_mc_approx_tilt = np.histogram(mc_approx_tilt_, bins=fill_bins)

    counts_mc_approx_tilt = raw_counts_mc_approx_tilt
    
    # fill x-error array as the bin sizes have changed
    xwidth = [(fill_bins[i+1]-fill_bins[i])/2 for i in range(len(fill_bins)-1)]
                        
    # MC errors - some ambiguity here, but i have decided to use the sqrt(raw counts), then scale the errors appropriately
    data_errors = np.sqrt(counts_data)
    mc_errors_approx_tilt = np.sqrt(raw_counts_mc_approx_tilt)
    
    # updated bincenters using the new binning
    binscenters = np.array([0.5 * (fill_bins[i] + fill_bins[i + 1]) for i in range(len(fill_bins) - 1)])
    
    # Compute chi-squared
    chi2_approx_tilt = np.sum(((np.array(counts_data) - np.array(counts_mc_approx_tilt)) ** 2) / 
              (np.array(data_errors) ** 2 + np.array(mc_errors_approx_tilt) ** 2))
    ndf = len(counts_data) - 1  # degrees of freedom (can be adjusted)
    print(f"\nApprox tilt Chi2/NDF: {chi2_approx_tilt / ndf:.2f}")
    
    
    # plot
    
    ax.set_title(labels[sp], fontsize=12)
    ax.set_xlabel('Cluster charge [p.e.]')
    ax.set_ylabel('clusters / ' + str(bin_size) + ' p.e. bin', fontsize = 12)
    ax.set_xlim([0, 80])  # default
    #ax.set_xlim([0, 35])  # WB/HM
    
    # MC histogram (approx-tilted)
    ax.hist(binscenters, bins=fill_bins, weights=counts_mc_approx_tilt, histtype='step', color=geo_colors, linewidth=2)
    ax.fill_between(      # shaded MC error band (turn off if needed)
        binscenters,
        counts_mc_approx_tilt - mc_errors_approx_tilt,
        counts_mc_approx_tilt + mc_errors_approx_tilt,
        step='mid',
        color=geo_colors,
        alpha=0.3
    )
    
    ax.errorbar(binscenters, counts_data, yerr=data_errors, fmt='o', color='k', label='Data')
    #ax.text(0.6, 0.85, f"$\chi^2$/ndf = {round(chi2)} / {int(ndf)}", transform=ax.transAxes, fontsize=8, color='purple')
    ax.text(0.6, 0.85,"entries = " + str(sum(counts_data)), fontsize = 9,transform = ax.transAxes)
    ax.text(0.6, 0.75, f"$\chi^2$/ndf = {round(chi2_approx_tilt)} / {int(ndf)}", transform = ax.transAxes, fontsize=9, color=geo_colors)
    

# Hide empty subplots
for i in range(sp + 1, len(axes)):
    fig.delaxes(axes[i])
    
# Custom legend entries
legend_elements = [
    Line2D([0], [0], color='k', marker='o', linestyle='None', label='Data'),
    Line2D([0], [0], color=geo_colors, linestyle='-', label='MC'),
]

# Place global legend
fig.legend(handles=legend_elements, loc='upper center', ncol=4, fontsize=24, frameon=False, bbox_to_anchor=(0.5, 1.05))

plt.tight_layout()
path_str = 'plots/geometry_comparison/AmBe/approx_geo_agreement/AmBe_total_cluster_pe_all_source_positions_ApproxTilt_QE_1.50_HM_1.25.png'
#plt.savefig(path_str,dpi=300,bbox_inches='tight',pad_inches=.3,facecolor = 'w')
plt.show() 



############################
Port 5, Y = +100cm

Approx tilt Chi2/NDF: 0.88


############################
Port 5, Y = +50cm

Approx tilt Chi2/NDF: 4.14


############################
Port 5, Y = 0cm

Approx tilt Chi2/NDF: 0.89


############################
Port 5, Y = -50cm

Approx tilt Chi2/NDF: 2.87


############################
Port 5, Y = -100cm

Approx tilt Chi2/NDF: 1.81


############################
Port 1, Y = +100cm

Approx tilt Chi2/NDF: 1.89


############################
Port 1, Y = +50cm

Approx tilt Chi2/NDF: 2.73


############################
Port 1, Y = 0cm

Approx tilt Chi2/NDF: 2.03


############################
Port 1, Y = -50cm

Approx tilt Chi2/NDF: 2.00


############################
Port 1, Y = -100cm

Approx tilt Chi2/NDF: 7.07


############################
Port 4, Y = +100cm

Approx tilt Chi2/NDF: 1.65


############################
Port 4, Y = +50cm

Approx tilt Chi2/NDF: 1.30


############################
Port 4, Y = 0cm

Approx tilt Chi2/NDF: 1.61


### Sum across ports for each depth

In [41]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.gridspec as gridspec
from scipy.stats import chi2_contingency
import random
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches

# new labels for depths (sum over port positions)
depths_cm = [+100, +50, 0, -50, -100]
depth_labels = [f"Y = {y} cm" for y in depths_cm]
depth_groups = {y: [] for y in depths_cm}
for i, p in enumerate(poss):
    depth_groups[p[1]].append(i)
    
MC_poss = [[0,100,0,0],[0,50,0,0],[0,0,0,0],[0,-50,0,0],[0,-100,0,0],            # Port 5
          [0,100,-75,0],[0,50,-75,0],[0,0,-75,0],[0,-50,-75,0],[0,-100,-75,0],   # Port 1
          [75,100,0,0],[75,50,0,0],[75,0,0,0],[75,-50,0,0],[75,-100,0,0],   # Port 4
          [0,100,102,0],[0,50,102,0],[0,0,102,0],[0,-50,102,0],[0,-100,102,0]]   # Port 3
    

# Define number of rows and columns for the grid layout
fig = plt.figure(figsize=(20, 4))  # Adjust as needed
outer_gs = gridspec.GridSpec(1, 5, wspace=0.5)  # 5 columns for 5 depths

for d, y in enumerate(depths_cm):   # 5 plots in total, one for each depth
    
    # default
    bin_size = 2
    binning = np.arange(5, 71, bin_size)
    
    # Watchboy/HM
    #bin_size = 1
    #binning = np.arange(0, 31, bin_size)

    print('\n############################')
    print(depth_labels[d], '\n')
    
    data_entries_all = []
    mc_entries_approx_all = []

    # Prepare samples with equal counts
    for sp in depth_groups[y]:
        
        if MC_poss[sp][0] == 0 and MC_poss[sp][1] == 100 and MC_poss[sp][2] == 102:
            print('\nPort 3 +100cm NOT INCLUDED')
            continue
        
        data_entries = []
        mc_approx_tilt_entries = []
        
        data_entries = [cluster_charge[i] for i in range(len(cluster_charge)) if 
             source_position[0][i] == poss[sp][0] and
             source_position[1][i] == poss[sp][1] and
             source_position[2][i] == poss[sp][2]]

        mc_approx_tilt_entries = [cluster_charge_MC[i] for i in range(len(cluster_charge_MC)) if 
                   source_position_MC[0][i] == MC_poss[sp][0] and 
                   source_position_MC[1][i] == MC_poss[sp][1] and
                   source_position_MC[2][i] == MC_poss[sp][2]]
        
    
        if len(data_entries) > 0 or len(mc_approx_tilt_entries):
            data_entries_all.append(data_entries)
            mc_entries_approx_all.append(mc_approx_tilt_entries)
            
        
    # Randomly subsample per-position to ensure balanced contribution
    data_ = []
    mc_approx_tilt_ = []

    for data_entries, mc_entries_approx in zip(data_entries_all, mc_entries_approx_all):
        nmin = min(len(data_entries), len(mc_entries_approx))
        if nmin == 0:
            continue
        data_ += random.sample(data_entries, nmin)
        mc_approx_tilt_ += random.sample(mc_entries_approx, nmin)

        
    assert len(data_) == len(mc_approx_tilt_), f"Mismatch at depth {y} cm"
        
    
    # MC counts and bins
    counts_data, bins_data = np.histogram(data_, bins=binning)
    raw_counts_mc_approx_tilt, bins_mc_approx_tilt = np.histogram(mc_approx_tilt_, bins=binning)

    counts_mc_approx_tilt = raw_counts_mc_approx_tilt

    fill_bins = generate_merged_bins(binning, counts_data, counts_mc_approx_tilt, min_counts=10)

    # re-histogram with updated binning
    counts_data, bins_data = np.histogram(data_, bins=fill_bins)
    raw_counts_mc_approx_tilt, bins_mc_approx_tilt = np.histogram(mc_approx_tilt_, bins=fill_bins)

    counts_mc_approx_tilt = raw_counts_mc_approx_tilt
    
    # fill x-error array as the bin sizes have changed
    xwidth = [(fill_bins[i+1]-fill_bins[i])/2 for i in range(len(fill_bins)-1)]
                        
    # MC errors - some ambiguity here, but i have decided to use the sqrt(raw counts), then scale the errors appropriately
    data_errors = np.sqrt(counts_data)
    mc_errors_approx_tilt = np.sqrt(raw_counts_mc_approx_tilt)
    
    # updated bincenters using the new binning
    binscenters = np.array([0.5 * (fill_bins[i] + fill_bins[i + 1]) for i in range(len(fill_bins) - 1)])
    
    # Compute chi-squared
    chi2_approx_tilt = np.sum(((np.array(counts_data) - np.array(counts_mc_approx_tilt)) ** 2) / 
              (np.array(data_errors) ** 2 + np.array(mc_errors_approx_tilt) ** 2))
    ndf = len(counts_data) - 1  # degrees of freedom (can be adjusted)
    print(f"\nApprox tilt Chi2/NDF: {chi2_approx_tilt / ndf:.2f}")
    
    
    # calculate the residuals
    mc_approx_tilt_data = []; mc_errors_approx_tilt_plot = []; 
    x_info = []

    for i in range(len(counts_data)):
        if counts_data[i] == 0:
            continue  # skip bin with no data
        x_info.append(binscenters[i])
        
        # MC (approx-tilted)
        if counts_mc_approx_tilt[i] == 0:
            mc_approx_tilt_data.append(0)
            mc_errors_approx_tilt_plot.append(0)
        else:
            ratio_mc_approx_tilt = counts_mc_approx_tilt[i] / counts_data[i]
            error_mc_approx_tilt = ratio_mc_approx_tilt * np.sqrt(
                (data_errors[i] / counts_data[i])**2 +
                (mc_errors_approx_tilt[i] / counts_mc_approx_tilt[i])**2
            )
            mc_approx_tilt_data.append(ratio_mc_approx_tilt)
            mc_errors_approx_tilt_plot.append(error_mc_approx_tilt)
    
    
    inner_gs = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=outer_gs[d], height_ratios=[3, 1], hspace=0.35)
    ax1 = fig.add_subplot(inner_gs[0])  # main histograms
    ax2 = fig.add_subplot(inner_gs[1])  # residuals

    
    ax1.set_title(depth_labels[d], fontsize=12)
    ax1.set_ylabel('clusters / ' + str(bin_size) + ' p.e. bin', fontsize = 12)
    ax1.set_xlim([0, 80])  # default
    #ax1.set_xlim([0, 35])   # WB / HM

    # MC histogram (approx-tilted)
    ax1.hist(binscenters, bins=fill_bins, weights=counts_mc_approx_tilt, histtype='step', color=geo_colors, linewidth=2)
    ax1.fill_between(      # shaded MC error band (turn off if needed)
        binscenters,
        counts_mc_approx_tilt - mc_errors_approx_tilt,
        counts_mc_approx_tilt + mc_errors_approx_tilt,
        step='mid',
        color=geo_colors,
        alpha=0.3
    )
    
    ax1.errorbar(binscenters, counts_data, yerr=data_errors, fmt='o', color='k', label='Data')
    ax1.text(0.54, 0.9, f"events = {sum(counts_data)}", transform = ax1.transAxes, fontsize=8)
    ax1.text(0.54, 0.8, f"$\chi^2$/ndf = {round(chi2_approx_tilt)} / {int(ndf)}", transform = ax1.transAxes, fontsize=8, color=geo_colors)
    
    # Residual subplot
    ax2.errorbar(x_info, mc_approx_tilt_data, xerr=xwidth, yerr=mc_errors_approx_tilt_plot, fmt='o', color=geo_colors, markersize=3)
    ax2.axhline(1, linestyle='dashed', color='k')
    ax2.set_ylim([0, 2])
    ax2.set_xlim([0, 80])  # default
    #ax2.set_xlim([0, 35])  # WB / HM
    ax2.set_ylabel("MC / Data", fontsize=10)
    ax2.set_xlabel("cluster charge [p.e.]", fontsize=12)
    
    ax1.xaxis.set_major_locator(MultipleLocator(20))  # ticks every 5 units of pe
    ax2.xaxis.set_major_locator(MultipleLocator(20))
    
    
# Custom legend entries
legend_elements = [
    Line2D([0], [0], color='k', marker='o', linestyle='None', label='Data'),
    Line2D([0], [0], color=geo_colors, linestyle='-', label='MC'),
]

# Place global legend
fig.legend(handles=legend_elements, loc='upper center', ncol=4, fontsize=24, frameon=False, bbox_to_anchor=(0.5, -0.05))
    
plt.tight_layout()
path_str = 'plots/geometry_comparison/AmBe/approx_geo_agreement/AmBe_total_cluster_pe_ApproxTilt_QE_1.50_HM_1.25.png'
plt.savefig(path_str,dpi=300,bbox_inches='tight',pad_inches=.3,facecolor = 'w')
plt.show()


############################
Y = 100 cm 


Port 3 +100cm NOT INCLUDED

Approx tilt Chi2/NDF: 1.75

############################
Y = 50 cm 


Approx tilt Chi2/NDF: 3.56

############################
Y = 0 cm 


Approx tilt Chi2/NDF: 2.81

############################
Y = -50 cm 


Approx tilt Chi2/NDF: 2.11

############################
Y = -100 cm 


Approx tilt Chi2/NDF: 2.74


/var/folders/nf/wq25_wtn08vb7t3wg0hydmhr0000gr/T/ipykernel_3247/813050863.py:184: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


In [42]:
'''Same as above but no plotting - just run the chi sq fitting N times for an average / median value'''

x2_approx = [[], [], [], [], []]


for jx in trange(100):
    index = 0
    for d, y in enumerate(depths_cm):   # 5 plots in total, one for each depth

        # default
        bin_size = 2
        binning = np.arange(5, 71, bin_size)

        # Watchboy/HM
        #bin_size = 1
        #binning = np.arange(0, 31, bin_size)

        #print('\n############################')
        #print(depth_labels[d], '\n')

        data_entries_all = []
        mc_entries_approx_all = []

        # Prepare samples with equal counts
        for sp in depth_groups[y]:
            
            if MC_poss[sp][0] == 0 and MC_poss[sp][1] == 100 and MC_poss[sp][2] == 102:
                #print('\nPort 3 +100cm NOT INCLUDED')
                continue

            data_entries = []
            mc_approx_tilt_entries = []

            data_entries = [cluster_charge[i] for i in range(len(cluster_charge)) if 
                 source_position[0][i] == poss[sp][0] and
                 source_position[1][i] == poss[sp][1] and
                 source_position[2][i] == poss[sp][2]]

            mc_approx_tilt_entries = [cluster_charge_MC[i] for i in range(len(cluster_charge_MC)) if 
                       source_position_MC[0][i] == MC_poss[sp][0] and 
                       source_position_MC[1][i] == MC_poss[sp][1] and
                       source_position_MC[2][i] == MC_poss[sp][2]]


            if len(data_entries) > 0 or len(mc_approx_tilt_entries):
                data_entries_all.append(data_entries)
                mc_entries_approx_all.append(mc_approx_tilt_entries)


        # Randomly subsample per-position to ensure balanced contribution
        data_ = []
        mc_approx_tilt_ = []

        for data_entries,mc_entries_approx in zip(data_entries_all, mc_entries_approx_all):
            nmin = min(len(data_entries), len(mc_entries_approx))
            if nmin == 0:
                continue
            data_ += random.sample(data_entries, nmin)
            mc_approx_tilt_ += random.sample(mc_entries_approx, nmin)


        assert len(data_) == len(mc_approx_tilt_), f"Mismatch at depth {y} cm"


        # MC counts and bins
        counts_data, bins_data = np.histogram(data_, bins=binning)
        raw_counts_mc_approx_tilt, bins_mc_approx_tilt = np.histogram(mc_approx_tilt_, bins=binning)

        counts_mc_approx_tilt = raw_counts_mc_approx_tilt

        fill_bins = generate_merged_bins(binning, counts_data, counts_mc_approx_tilt, min_counts=10)

        # re-histogram with updated binning
        counts_data, bins_data = np.histogram(data_, bins=fill_bins)
        raw_counts_mc_approx_tilt, bins_mc_approx_tilt = np.histogram(mc_approx_tilt_, bins=fill_bins)

        counts_mc_approx_tilt = raw_counts_mc_approx_tilt

        # fill x-error array as the bin sizes have changed
        xwidth = [(fill_bins[i+1]-fill_bins[i])/2 for i in range(len(fill_bins)-1)]

        # MC errors - some ambiguity here, but i have decided to use the sqrt(raw counts), then scale the errors appropriately
        data_errors = np.sqrt(counts_data)
        mc_errors_approx_tilt = np.sqrt(raw_counts_mc_approx_tilt)

        # updated bincenters using the new binning
        binscenters = np.array([0.5 * (fill_bins[i] + fill_bins[i + 1]) for i in range(len(fill_bins) - 1)])

        # Compute chi-squared
        chi2_approx_tilt = np.sum(((np.array(counts_data) - np.array(counts_mc_approx_tilt)) ** 2) / 
                  (np.array(data_errors) ** 2 + np.array(mc_errors_approx_tilt) ** 2))
        ndf = len(counts_data) - 1  # degrees of freedom (can be adjusted)
        #print(f"\nNo tilt Chi2/NDF: {chi2_no_tilt / ndf:.2f}")
        #print(f"Indy tilt Chi2/NDF: {chi2_indy_tilt / ndf:.2f}")
        #print(f"Approx tilt Chi2/NDF: {chi2_approx_tilt / ndf:.2f}")

        x2_approx[index].append(chi2_approx_tilt / ndf)
        index += 1
        
    
print('\ndone')

100%|█████████████████████████████████████████| 100/100 [01:53<00:00,  1.13s/it]


done


In [15]:
# chisq sampling per position (source and depth)

import matplotlib.pyplot as plt
import numpy as np
import matplotlib.gridspec as gridspec
from scipy.stats import chi2_contingency
import random
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches

# [0,100,102,0] (remove Port 3 +100cm)

# labels to match source position (poss) list
labels = [
    "Port 5, Y = +100cm", "Port 5, Y = +50cm", "Port 5, Y = 0cm", "Port 5, Y = -50cm", "Port 5, Y = -100cm",
    "Port 1, Y = +100cm", "Port 1, Y = +50cm", "Port 1, Y = 0cm", "Port 1, Y = -50cm", "Port 1, Y = -100cm",
    "Port 4, Y = +100cm", "Port 4, Y = +50cm", "Port 4, Y = 0cm", "Port 4, Y = -50cm", "Port 4, Y = -100cm",
    "Port 3, Y = +100cm", "Port 3, Y = +50cm", "Port 3, Y = 0cm", "Port 3, Y = -50cm", "Port 3, Y = -100cm"
]

MC_poss = [[0,100,0,0],[0,50,0,0],[0,0,0,0],[0,-50,0,0],[0,-100,0,0],            # Port 5
          [0,100,-75,0],[0,50,-75,0],[0,0,-75,0],[0,-50,-75,0],[0,-100,-75,0],   # Port 1
          [75,100,0,0],[75,50,0,0],[75,0,0,0],[75,-50,0,0],[75,-100,0,0],        # Port 4
          [0,100,102,0], [0,50,102,0],[0,0,102,0],[0,-50,102,0],[0,-100,102,0]]  # Port 3


def generate_merged_bins(bins, counts_data, counts_mc_1, min_counts=10):
    """
    Given initial bin edges and counts for data and MC, merge adjacent bins
    until each has at least `min_counts` in both histograms.

    Returns a new list of bin edges (fill_bins).
    """
    assert len(bins) == len(counts_data) + 1, "Bin edge mismatch"

    new_bins = [bins[0]]  # always start with the first edge
    i = 0
    while i < len(counts_data):
        # Start with current bin
        sum_d = counts_data[i]
        sum_mc_1 = counts_mc_1[i]
        j = i + 1

        # Merge forward until we meet minimum count requirement
        while (sum_d < min_counts or sum_mc_1 < min_counts) and j < len(counts_data):
            sum_d += counts_data[j]
            sum_mc_1 += counts_mc_1[j]
            j += 1

        # Append right edge of merged bin
        new_bins.append(bins[j])
        i = j  # Move to next unprocessed bin

    return new_bins


x2_approx = [[], [], [], [], [],      # 5
            [], [], [], [], [],       # 1
            [], [], [], [], [],       # 4
            [], [], [], [], []]       # 3 (+100 will be empty)

for jk in range(100):  # 100 trials
    print('TRIAL ' + str(jk+1))
    for sp in range(len(poss)):

        # default
        bin_size = 4
        binning = np.arange(5, 69, bin_size)

        # Watchboy/HM
        #bin_size = 2
        #binning = np.arange(0, 28, bin_size)

        #print('\n\n############################')
        #print(labels[sp])

        # we don't want a figure for Port 3 +100cm --> keep it blank
        if labels[sp] == "Port 3, Y = +100cm":
            ax.axis("off")  # keep blank
            continue

        # we can extract the full data and MC samples for this position
        data_ = [cluster_charge[i] for i in range(len(cluster_charge)) if 
                 source_position[0][i] == poss[sp][0] and
                 source_position[1][i] == poss[sp][1] and
                 source_position[2][i] == poss[sp][2]]

        mc_approx_tilt_ = [cluster_charge_MC[i] for i in range(len(cluster_charge_MC)) if 
                   source_position_MC[0][i] == MC_poss[sp][0] and 
                   source_position_MC[1][i] == MC_poss[sp][1] and
                   source_position_MC[2][i] == MC_poss[sp][2]]


        # then equalize entries per position by random sampling
        nmin = min(len(data_), len(mc_approx_tilt_))
        if nmin == 0:
            continue  # Skip if no valid entries

        # Randomly sample equal-sized subsets
        data_ = random.sample(data_, nmin)
        mc_approx_tilt_ = random.sample(mc_approx_tilt_, nmin)

        #print(len(data_), len(mc_no_tilt_), len(mc_indy_tilt_), len(mc_approx_tilt_))
        if len(data_) == 0 or len(mc_approx_tilt_) == 0:
            continue  # Skip if no data


        # MC counts and bins
        counts_data, bins_data = np.histogram(data_, bins=binning)
        raw_counts_mc_approx_tilt, bins_mc_approx_tilt = np.histogram(mc_approx_tilt_, bins=binning)

        counts_mc_approx_tilt = raw_counts_mc_approx_tilt

        fill_bins = generate_merged_bins(binning, counts_data, counts_mc_approx_tilt, min_counts=10)

        # re-histogram with updated binning
        counts_data, bins_data = np.histogram(data_, bins=fill_bins)
        raw_counts_mc_approx_tilt, bins_mc_approx_tilt = np.histogram(mc_approx_tilt_, bins=fill_bins)

        counts_mc_approx_tilt = raw_counts_mc_approx_tilt

        # fill x-error array as the bin sizes have changed
        xwidth = [(fill_bins[i+1]-fill_bins[i])/2 for i in range(len(fill_bins)-1)]

        # MC errors - some ambiguity here, but i have decided to use the sqrt(raw counts), then scale the errors appropriately
        data_errors = np.sqrt(counts_data)
        mc_errors_approx_tilt = np.sqrt(raw_counts_mc_approx_tilt)

        # updated bincenters using the new binning
        binscenters = np.array([0.5 * (fill_bins[i] + fill_bins[i + 1]) for i in range(len(fill_bins) - 1)])

        # Compute chi-squared
        chi2_approx_tilt = np.sum(((np.array(counts_data) - np.array(counts_mc_approx_tilt)) ** 2) / 
                  (np.array(data_errors) ** 2 + np.array(mc_errors_approx_tilt) ** 2))
        ndf = len(counts_data) - 1  # degrees of freedom (can be adjusted)
        x2_approx[sp].append(chi2_approx_tilt / ndf)


print('\ndone')

TRIAL 1
TRIAL 2
TRIAL 3
TRIAL 4
TRIAL 5
TRIAL 6
TRIAL 7
TRIAL 8
TRIAL 9
TRIAL 10
TRIAL 11
TRIAL 12
TRIAL 13
TRIAL 14
TRIAL 15
TRIAL 16
TRIAL 17
TRIAL 18
TRIAL 19
TRIAL 20
TRIAL 21
TRIAL 22
TRIAL 23
TRIAL 24
TRIAL 25
TRIAL 26
TRIAL 27
TRIAL 28
TRIAL 29
TRIAL 30
TRIAL 31
TRIAL 32
TRIAL 33
TRIAL 34
TRIAL 35
TRIAL 36
TRIAL 37
TRIAL 38
TRIAL 39
TRIAL 40
TRIAL 41
TRIAL 42
TRIAL 43
TRIAL 44
TRIAL 45
TRIAL 46
TRIAL 47
TRIAL 48
TRIAL 49
TRIAL 50
TRIAL 51
TRIAL 52
TRIAL 53
TRIAL 54
TRIAL 55
TRIAL 56
TRIAL 57
TRIAL 58
TRIAL 59
TRIAL 60
TRIAL 61
TRIAL 62
TRIAL 63
TRIAL 64
TRIAL 65
TRIAL 66
TRIAL 67
TRIAL 68
TRIAL 69
TRIAL 70
TRIAL 71
TRIAL 72
TRIAL 73
TRIAL 74
TRIAL 75
TRIAL 76
TRIAL 77
TRIAL 78
TRIAL 79
TRIAL 80
TRIAL 81
TRIAL 82
TRIAL 83
TRIAL 84
TRIAL 85
TRIAL 86
TRIAL 87
TRIAL 88
TRIAL 89
TRIAL 90
TRIAL 91
TRIAL 92
TRIAL 93
TRIAL 94
TRIAL 95
TRIAL 96
TRIAL 97
TRIAL 98
TRIAL 99
TRIAL 100
done


In [26]:
# chi_sq plot (summed depths)

# Example depth values (from top to bottom, increasing Y means deeper)
depths_cm = [+100, +50, 0, -50, -100]    # depths_cm = [+100, +50, 0, -50, -100]
chisq_dof_approx_tilt = []; chisq_dof_approx_tilt_er = []
for i in range(len(x2_approx)):
    chisq_dof_approx_tilt.append(np.median(x2_approx[i]))   # list of median values sampled
    chisq_dof_approx_tilt_er.append(np.std(x2_approx[i]))
    
# for all
median_approx_tilt = 1.74

# Watchboys
#median_no_tilt = 32.37
#median_approx_tilt = 5.15


plt.figure(figsize=(4, 3))
#plt.figure(figsize=(5, 3))

plt.axhline(median_approx_tilt, linestyle = 'dashed', color = 'black', label = 'Total fit')

print('PORT 5:', labels[0:5], '\n')
print('PORT 1:', labels[5:10], '\n')
print('PORT 4:', labels[10:15], '\n')
print('PORT 3:', labels[16:20], '\n')

plt.errorbar(
    depths_cm, chisq_dof_approx_tilt[0:5], yerr=chisq_dof_approx_tilt_er[0:5],
    fmt='o-', color='tab:orange', capsize=3, label = 'Port 5'
)
plt.errorbar(
    depths_cm, chisq_dof_approx_tilt[5:10], yerr=chisq_dof_approx_tilt_er[5:10],
    fmt='o-', color='tab:red', capsize=3, label = 'Port 1'
)
plt.errorbar(
    depths_cm, chisq_dof_approx_tilt[10:15], yerr=chisq_dof_approx_tilt_er[10:15],
    fmt='o-', color='tab:blue', capsize=3, label = 'Port 4'
)
plt.errorbar(
    depths_cm[1:5], chisq_dof_approx_tilt[16:20], yerr=chisq_dof_approx_tilt_er[16:20],
    fmt='o-', color='tab:green', capsize=3, label = 'Port 3'
)


plt.xlabel('Source Depth [cm]')
plt.ylabel(r'$\chi^{2} / dof$')
plt.title('AmBe neutrons')
plt.legend(fontsize = 12, frameon = False, ncol = 2)
#plt.legend(fontsize=10, frameon=False, loc='center left', bbox_to_anchor=(1, 0.75))
plt.ylim([0, 10])
plt.gca().invert_xaxis()  # to align with above plots
plt.tight_layout()
path_str = 'plots/geometry_comparison/AmBe/approx_geo_agreement/100_TRIALS_ALL_POS_CHISQ_ApproxTilt_AmBe_total_cluster_pe_QE_1.50_HM_1.25.png'
plt.savefig(path_str,dpi=300,bbox_inches='tight',pad_inches=.3,facecolor = 'w')
plt.show()


print('done')

PORT 5: ['Port 5, Y = +100cm', 'Port 5, Y = +50cm', 'Port 5, Y = 0cm', 'Port 5, Y = -50cm', 'Port 5, Y = -100cm'] 

PORT 1: ['Port 1, Y = +100cm', 'Port 1, Y = +50cm', 'Port 1, Y = 0cm', 'Port 1, Y = -50cm', 'Port 1, Y = -100cm'] 

PORT 4: ['Port 4, Y = +100cm', 'Port 4, Y = +50cm', 'Port 4, Y = 0cm', 'Port 4, Y = -50cm', 'Port 4, Y = -100cm'] 

PORT 3: ['Port 3, Y = +50cm', 'Port 3, Y = 0cm', 'Port 3, Y = -50cm', 'Port 3, Y = -100cm'] 

done


In [9]:
# chi_sq plot (summed depths)

# Example depth values (from top to bottom, increasing Y means deeper)
depths_cm = [+100, +50, 0, -50, -100]    # depths_cm = [+100, +50, 0, -50, -100]
    
# for all
median_approx_tilt = 1.74

# Watchboys
#median_no_tilt = 32.37
#median_approx_tilt = 5.15


plt.figure(figsize=(4, 3))
#plt.figure(figsize=(5, 3))

#plt.axhline(median_approx_tilt, linestyle = 'dashed', color = geo_colors)

plt.plot(depths_cm, [20/14, 54/14, 9/14, 31/13, 26/13],
         marker='o', linestyle='-', color = 'tab:orange', label = 'Port 5')
plt.plot(depths_cm, [45/14, 43/14, 36/14, 38/14, 84/14],
         marker='o', linestyle='-', color = 'tab:red', label = 'Port 1')
plt.plot(depths_cm, [35/14, 35/14, 24/14, 17/14, 27/13],
         marker='o', linestyle='-', color = 'tab:blue', label = 'Port 4')
plt.plot([50, 0, -50, -100], [28/14, 36/13, 24/13, 25/12],
         marker='o', linestyle='-', color = 'tab:green', label = 'Port 3')

#plt.errorbar(
#    depths_cm, [1,2,1,2,1], yerr=[0.1,0.1,0.1,0.1,0.1],
#    fmt='o-', color='red', capsize=3
#)


plt.xlabel('Source Depth [cm]')
plt.ylabel(r'$\chi^{2} / dof$')
plt.title('AmBe neutrons')
plt.legend(fontsize = 12, frameon = False)
#plt.legend(fontsize=10, frameon=False, loc='center left', bbox_to_anchor=(1, 0.75))
plt.ylim([0, 10])
#plt.ylim([0, 10])
plt.gca().invert_xaxis()  # to align with above plots
plt.tight_layout()
path_str = 'plots/geometry_comparison/AmBe/approx_geo_agreement/POS_SUM_BY_DEPTH_CHISQ_ApproxTilt_AmBe_total_cluster_pe_QE_1.50_HM_1.25.png'
#plt.savefig(path_str,dpi=300,bbox_inches='tight',pad_inches=.3,facecolor = 'w')
plt.show()


print('done')

done
